# 01 — Data Cleaning (DeliveryETA)

Ce notebook reprend **étape par étape** les actions du script `src/preprocessing/cleaning.py` (classe `DataCleaner`), avec pour chaque étape :

- **Action** : ce que fait le code
- **Justification** : pourquoi on le fait, et ce que montrent les résultats

**Ordre du pipeline :** chargement → inspection → nettoyage du texte → conversion des types → valeurs manquantes → outliers → doublons → validation → sauvegarde.

## 0. Imports et chemins

In [41]:
import pandas as pd

RAW_PATH = "../data/dataset_delivery_time.csv"
CLEAN_PATH = "../data/dataset_delivery_time_clean.csv"


**Action :** lecture du CSV brut avec `pd.read_csv`.


In [42]:
df = pd.read_csv(CLEAN_PATH)
df.head()

,Time_Order_picked,Weatherconditions,Road_traffic_density,Type_of_vehicle,Time_taken(min),Distance_km
0,11:45:00,Sunny,High,motorcycle,24,3.025149
1,19:50:00,Stormy,Jam,scooter,33,20.183530
2,08:45:00,Sandstorms,Low,motorcycle,26,1.552758
3,18:10:00,Sunny,Medium,motorcycle,21,7.790401
4,13:45:00,Cloudy,High,scooter,30,6.210138



**Action :** Aficher le structure du dataset


In [28]:
print("Dimensions :", df.shape)

print("\nColonnes :")
print(df.columns.tolist())

print("\nTypes et valeurs non nulles :")
df.info()

print("\nStatistiques numériques :")
print(df.describe())

print("data typse")
for column in df.columns:
    print(f"{column} : {df[column].dtype}")

Dimensions : (41934, 9)

Colonnes :
['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle', 'Time_taken(min)']

Types et valeurs non nulles :
<class 'pandas.DataFrame'>
RangeIndex: 41934 entries, 0 to 41933
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Restaurant_latitude          41934 non-null  float64
 1   Restaurant_longitude         41934 non-null  float64
 2   Delivery_location_latitude   41934 non-null  float64
 3   Delivery_location_longitude  41934 non-null  float64
 4   Time_Order_picked            41934 non-null  str    
 5   Weatherconditions            41934 non-null  str    
 6   Road_traffic_density         41934 non-null  str    
 7   Type_of_vehicle              41934 non-null  str    
 8   Time_taken(min)              41934 non

**Action :** afficher les valeurs manquantes


In [29]:
missing = df.isna().sum()
print(missing)

Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Type_of_vehicle                0
Time_taken(min)                0
dtype: int64



**Action :** aficher le number DOUBLONS

In [37]:
duplicates = df.duplicated().sum()

print("\n DOUBLONS ")
print("Nombre de doublons :", duplicates)


 DOUBLONS 
Nombre de doublons : 0


**action :** aficher les category in Weatherconditions

In [33]:
print(df['Weatherconditions'].unique())

<ArrowStringArray>
['Sunny', 'Stormy', 'Sandstorms', 'Cloudy', 'Fog', 'Windy']
Length: 6, dtype: str


In [34]:
print(df['Road_traffic_density'].unique())

<ArrowStringArray>
['High', 'Jam', 'Low', 'Medium']
Length: 4, dtype: str


In [45]:
print(df.describe(include='all'))

       Time_Order_picked Weatherconditions Road_traffic_density  \
count              41934             41934                41934   
unique               193                 6                    4   
top             21:30:00               Fog                  Low   
freq                 462              7578                14747   
mean                 NaN               NaN                  NaN   
std                  NaN               NaN                  NaN   
min                  NaN               NaN                  NaN   
25%                  NaN               NaN                  NaN   
50%                  NaN               NaN                  NaN   
75%                  NaN               NaN                  NaN   
max                  NaN               NaN                  NaN   

       Type_of_vehicle  Time_taken(min)   Distance_km  
count            41934     41934.000000  41934.000000  
unique               4              NaN           NaN  
top         motorcycle     

## 2 OUTLIERS
**Action :** aficher le numbre outliers in all column nimerique

In [52]:
numeric_cols = [
    "Distance_km",
    "Time_taken(min)"
]

outliers = {}

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    mask = (
        (df[col] < lower_bound)
        |
        (df[col] > upper_bound)
    )

    count = mask.sum()

    outliers[col] = count

    print(f"\n{col}")
    print(f"Q1          : {Q1}")
    print(f"Q3          : {Q3}")
    print(f"IQR         : {IQR}")
    print(f"Lower bound : {lower_bound}")
    print(f"Upper bound : {upper_bound}")
    print(f"Outliers    : {count}")

target_outliers = df[df['Time_taken(min)'] > 51.5]

print(target_outliers['Time_taken(min)'].value_counts().sort_index())


Distance_km
Q1          : 4.6576549370601725
Q3          : 13.68092007746041
IQR         : 9.023265140400238
Lower bound : -8.877242773540184
Upper bound : 27.215817788060768
Outliers    : 0

Time_taken(min)
Q1          : 19.0
Q3          : 32.0
IQR         : 13.0
Lower bound : -0.5
Upper bound : 51.5
Outliers    : 247
Time_taken(min)
52    73
53    89
54    85
Name: count, dtype: int64
